<a href="https://colab.research.google.com/github/Amw839/Amw839/blob/main/02_clean_cityOfNewyYork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np

attendance = pd.read_csv("attendance_raw.csv", dtype=str)
demo       = pd.read_csv("demographics_raw.csv", dtype=str)
loc        = pd.read_json("locations_raw.json", dtype=str)
raw        = pd.read_csv("income_raw.csv", dtype=str)

def snake(df):
    """lowercase column names, spaces and symbols become _"""
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
    return df

def quality_report(df, name):
    print(f"--- {name} ---")
    print("rows:", len(df), "| duplicates:", df.duplicated().sum())
    print("nulls per column:\n", df.isna().sum()[df.isna().sum() > 0])

In [ ]:
print(raw.shape)    # should be (33774, 3)
raw.head(3)

(33774, 3)


,NAME,B19013_001E,zip code tabulation area
0,ZCTA5 00601,17526,00601
1,ZCTA5 00602,20260,00602
2,ZCTA5 00603,17703,00603


In [ ]:
snake(attendance).columns

Index(['school', 'schoolyear', 'month_code', 'calmonth', 'gradelevel',
       'gradesort', 'roster_count', 'absent', 'present', 'released'],
      dtype='object')

In [ ]:
def clean_attendance(df):
    df = snake(df)

    # 1. DBN/school number: remove spaces, make uppercase
    df["school"] = df["school"].str.strip().str.upper()

    # 2. Turn the count columns into numbers
    for col in ["present", "absent", "released"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 3. Remove exact duplicate rows
    df = df.drop_duplicates()

    # 4. Drop rows where present AND absent are both 0
    df = df[(df["present"] + df["absent"]) > 0]

    # 5. New column: attendance rate
    df["attendance_rate"] = df["present"] / (df["present"] + df["absent"])

    return df

quality_report(attendance, "attendance BEFORE")
att_clean = clean_attendance(attendance)
quality_report(att_clean, "attendance AFTER")
att_clean.head()

--- attendance BEFORE ---
rows: 246196 | duplicates: 0
nulls per column:
 Series([], dtype: int64)
--- attendance AFTER ---
rows: 246196 | duplicates: 0
nulls per column:
 Series([], dtype: int64)


,school,schoolyear,month_code,calmonth,gradelevel,gradesort,roster_count,absent,present,released,attendance_rate
0,01M015,2019-20,1,Sep,1,1,26,13,427,0,0.970455
1,01M015,2019-20,1,Sep,2,2,38,13,620,0,0.979463
2,01M015,2019-20,1,Sep,3,3,35,22,552,0,0.961672
3,01M015,2019-20,1,Sep,4,4,30,13,460,0,0.972516
4,01M015,2019-20,1,Sep,5,5,21,19,322,0,0.944282


In [ ]:
print(list(snake(attendance).columns))
snake(attendance).head(3)

['school', 'schoolyear', 'month_code', 'calmonth', 'gradelevel', 'gradesort', 'roster_count', 'absent', 'present', 'released']


,school,schoolyear,month_code,calmonth,gradelevel,gradesort,roster_count,absent,present,released
0,01M015,2019-20,1,Sep,1,1,26,13,427,0
1,01M015,2019-20,1,Sep,2,2,38,13,620,0
2,01M015,2019-20,1,Sep,3,3,35,22,552,0


In [ ]:
att_clean["attendance_rate"].describe()

,attendance_rate
count,246196.000000
mean,0.903682
std,0.073282
min,0.000000
25%,0.883211
50%,0.921986
75%,0.947846
max,1.000000


In [ ]:
def clean_income(df):
    df = snake(df).rename(columns={"b19013_001e": "median_income",
                                   "zip_code_tabulation_area": "zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    df["median_income"] = pd.to_numeric(df["median_income"], errors="coerce")
    df.loc[df["median_income"] < 0, "median_income"] = np.nan   # -666666666 = no data
    return df[["zip", "median_income"]]

quality_report(raw, "income BEFORE")
income = clean_income(raw)
quality_report(income, "income AFTER")
income.head()

--- income BEFORE ---
rows: 33774 | duplicates: 0
nulls per column:
 Series([], dtype: int64)
--- income AFTER ---
rows: 33774 | duplicates: 0
nulls per column:
 median_income    3153
dtype: int64


,zip,median_income
0,00601,17526.0
1,00602,20260.0
2,00603,17703.0
3,00606,19603.0
4,00610,22796.0


What each line does:

snake(df) makes the names lowercase with underscores, so B19013_001E becomes b19013_001e.
.rename(...) gives the columns clear names: median_income and zip.
.str.zfill(5) keeps zip codes at 5 digits, so 601 becomes 00601.
pd.to_numeric(...) turns income from text into a number.
df.loc[... < 0] = np.nan turns the Census “no data” code -666666666 into a blank.
return df[["zip", "median_income"]] keeps only the 2 columns you need and drops NAME.

In [ ]:
att_clean.to_parquet("attendance_clean.parquet", index=False)
income.to_parquet("income_clean.parquet", index=False)

In [ ]:
print(list(snake(demo).columns))
snake(demo).head(3)

['dbn', 'school_name', 'year', 'total_enrollment', 'grade_3k_pk_half_day_full_day', 'grade_k', 'grade_1', 'grade_2', 'grade_3', 'grade_4', 'grade_5', 'grade_6', 'grade_7', 'grade_8', 'grade_9', 'grade_10', 'grade_11', 'grade_12', 'female', 'female', 'male', 'male', 'asian', 'asian', 'black', 'black', 'hispanic', 'hispanic', 'multiple_race_categories_not_represented', 'multiple_race_categories_not_represented', 'white', 'white', 'students_with_disabilities', 'students_with_disabilities', 'english_language_learners', 'english_language_learners', 'poverty', 'poverty', 'economic_need_index']


,dbn,school_name,year,total_enrollment,grade_3k_pk_half_day_full_day,grade_k,grade_1,grade_2,grade_3,grade_4,...,multiple_race_categories_not_represented,white,white,students_with_disabilities,students_with_disabilities,english_language_learners,english_language_learners,poverty,poverty,economic_need_index
0,01M015,P.S. 015 Roberto Clemente,2015-16,176,14,32,33,39,23,17,...,1.7%,2,1.1%,60,34.1%,16,9.1%,149,84.7%,88.9%
1,01M015,P.S. 015 Roberto Clemente,2016-17,178,17,28,33,27,31,24,...,2.2%,4,2.2%,51,28.7%,12,6.7%,152,85.4%,88.2%
2,01M015,P.S. 015 Roberto Clemente,2017-18,190,17,28,32,33,23,31,...,1.1%,6,3.2%,49,25.8%,8,4.2%,161,84.7%,89.0%


In [ ]:
print(list(demo.columns)[18:24])

['# Female', '% Female', '# Male', '% Male', '# Asian', '% Asian']


In [ ]:
def clean_demographics(df):
    df = df.copy()

    # 1. Keep # and % meaning before snake() removes the symbols
    df.columns = (df.columns
                  .str.replace("#", "n ", regex=False)
                  .str.replace("%", "pct ", regex=False))
    df = snake(df)

    # 2. DBN: remove spaces, make uppercase
    df["dbn"] = df["dbn"].str.strip().str.upper()

    # 3. Percent columns: "84.7%" -> 0.847 ; hidden values like "Above 95%" -> blank + flag
    pct_cols = [c for c in df.columns if c.startswith("pct_")] + ["economic_need_index"]
    for col in pct_cols:
        text = df[col].astype(str).str.strip()
        df[col + "_hidden"] = text.str.contains("Above|Below", case=False)
        df[col] = pd.to_numeric(text.str.rstrip("%"), errors="coerce") / 100

    # 4. Count columns: text -> numbers
    count_cols = ["total_enrollment"] + [c for c in df.columns
                  if c.startswith("grade_") or c.startswith("n_")]
    for col in count_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 5. Year "2019-20" -> keep it, and add 2019 as a number
    df["school_year_start"] = df["year"].str[:4].astype(int)

    return df.drop_duplicates()

quality_report(demo, "demographics BEFORE")
demo_clean = clean_demographics(demo)
quality_report(demo_clean, "demographics AFTER")
demo_clean.head(3)

--- demographics BEFORE ---
rows: 9096 | duplicates: 0
nulls per column:
 Series([], dtype: int64)
--- demographics AFTER ---
rows: 9096 | duplicates: 0
nulls per column:
 total_enrollment                 1030
grade_3k_pk_half_day_full_day       4
grade_9                            48
grade_10                           41
grade_11                           15
grade_12                           11
n_female                          143
n_male                            165
n_asian                            83
n_black                            44
n_hispanic                        123
n_white                            59
n_english_language_learners         8
n_poverty                         905
pct_poverty                       905
economic_need_index               416
dtype: int64


,dbn,school_name,year,total_enrollment,grade_3k_pk_half_day_full_day,grade_k,grade_1,grade_2,grade_3,grade_4,...,pct_asian_hidden,pct_black_hidden,pct_hispanic_hidden,pct_multiple_race_categories_not_represented_hidden,pct_white_hidden,pct_students_with_disabilities_hidden,pct_english_language_learners_hidden,pct_poverty_hidden,economic_need_index_hidden,school_year_start
0,01M015,P.S. 015 Roberto Clemente,2015-16,176.0,14.0,32,33,39,23,17,...,False,False,False,False,False,False,False,False,False,2015
1,01M015,P.S. 015 Roberto Clemente,2016-17,178.0,17.0,28,33,27,31,24,...,False,False,False,False,False,False,False,False,False,2016
2,01M015,P.S. 015 Roberto Clemente,2017-18,190.0,17.0,28,32,33,23,31,...,False,False,False,False,False,False,False,False,False,2017


In [ ]:
print(list(demo.columns)[18:24])

['# Female', '% Female', '# Male', '% Male', '# Asian', '% Asian']


In [ ]:
def clean_demographics(df):
    df = df.copy()

    # 1. Keep # and % meaning before snake() removes the symbols
    df.columns = (df.columns
                  .str.replace("#", "n ", regex=False)
                  .str.replace("%", "pct ", regex=False))
    df = snake(df)

    # 2. DBN: remove spaces, make uppercase
    df["dbn"] = df["dbn"].str.strip().str.upper()

    # 3. Percent columns: "84.7%" -> 0.847 ; hidden values like "Above 95%" -> blank + flag
    pct_cols = [c for c in df.columns if c.startswith("pct_")] + ["economic_need_index"]
    for col in pct_cols:
        text = df[col].astype(str).str.strip()
        df[col + "_hidden"] = text.str.contains("Above|Below", case=False)
        df[col] = pd.to_numeric(text.str.rstrip("%"), errors="coerce") / 100

    # 4. Count columns: text -> numbers
    count_cols = ["total_enrollment"] + [c for c in df.columns
                  if c.startswith("grade_") or c.startswith("n_")]
    for col in count_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 5. Year "2019-20" -> keep it, and add 2019 as a number
    df["school_year_start"] = df["year"].str[:4].astype(int)

    return df.drop_duplicates()

quality_report(demo, "demographics BEFORE")
demo_clean = clean_demographics(demo)
quality_report(demo_clean, "demographics AFTER")
demo_clean.head(3)

--- demographics BEFORE ---
rows: 9096 | duplicates: 0
nulls per column:
 Series([], dtype: int64)
--- demographics AFTER ---
rows: 9096 | duplicates: 0
nulls per column:
 total_enrollment                 1030
grade_3k_pk_half_day_full_day       4
grade_9                            48
grade_10                           41
grade_11                           15
grade_12                           11
n_female                          143
n_male                            165
n_asian                            83
n_black                            44
n_hispanic                        123
n_white                            59
n_english_language_learners         8
n_poverty                         905
pct_poverty                       905
economic_need_index               416
dtype: int64


,dbn,school_name,year,total_enrollment,grade_3k_pk_half_day_full_day,grade_k,grade_1,grade_2,grade_3,grade_4,...,pct_asian_hidden,pct_black_hidden,pct_hispanic_hidden,pct_multiple_race_categories_not_represented_hidden,pct_white_hidden,pct_students_with_disabilities_hidden,pct_english_language_learners_hidden,pct_poverty_hidden,economic_need_index_hidden,school_year_start
0,01M015,P.S. 015 Roberto Clemente,2015-16,176.0,14.0,32,33,39,23,17,...,False,False,False,False,False,False,False,False,False,2015
1,01M015,P.S. 015 Roberto Clemente,2016-17,178.0,17.0,28,33,27,31,24,...,False,False,False,False,False,False,False,False,False,2016
2,01M015,P.S. 015 Roberto Clemente,2017-18,190.0,17.0,28,32,33,23,31,...,False,False,False,False,False,False,False,False,False,2017


In [ ]:
print(demo_clean.columns.duplicated().sum())          # should be 0
demo_clean[["dbn", "year", "n_poverty", "pct_poverty", "economic_need_index"]].head()

0


,dbn,year,n_poverty,pct_poverty,economic_need_index
0,01M015,2015-16,149.0,0.847,0.889
1,01M015,2016-17,152.0,0.854,0.882
2,01M015,2017-18,161.0,0.847,0.890
3,01M015,2018-19,147.0,0.845,0.888
4,01M015,2019-20,155.0,0.816,0.869


No duplicate names, and you now have pairs like n_female / pct_female.
pct_poverty shows 0.847 instead of 84.7%.
Any _hidden column that shows True marks a value the school hid for privacy.

In [ ]:
demo_clean.to_parquet("demographics_clean.parquet", index=False)

This table connects everything: it links each school (DBN) to its zip code, which is how attendance and demographics join to the Census income data. We’ll mainly need:

the DBN column (it may have another name, like before)
the zip code column, kept as 5-digit text
borough, latitude and longitude for the map in your dashboard

In [ ]:
print(list(snake(loc).columns))
snake(loc).head(3)

['fiscal_year', 'system_code', 'location_code', 'location_name', 'beds', 'managed_by_name', 'location_type_description', 'location_category_description', 'grades_text', 'grades_final_text', 'open_date', 'status_descriptions', 'primary_building_code', 'primary_address_line_1', 'state_code', 'x_coordinate', 'y_coordinate', 'longitude', 'latitude', 'community_district', 'council_district', 'census_tract', 'borough_block_lot', 'nta', 'nta_name', 'principal_name', 'principal_title', 'principal_phone_number', 'fax_number', 'geographical_district_code', 'administrative_district_code', 'administrative_district_name', 'community_school_sup_name', 'tier_3_support_location_name', 'tier_3_support_leader_name', 'tier_2_support_location_name', 'highschool_network_location', 'highschool_network_name', 'highschool_network', 'community_district_1', 'police_precinct']


,fiscal_year,system_code,location_code,location_name,beds,managed_by_name,location_type_description,location_category_description,grades_text,grades_final_text,...,administrative_district_name,community_school_sup_name,tier_3_support_location_name,tier_3_support_leader_name,tier_2_support_location_name,highschool_network_location,highschool_network_name,highschool_network,community_district_1,police_precinct
0,2020,15K001,K001,P.S. 001 The Bergen,331500010001,DOE,General Academic,Elementary,"PK,0K,01,02,03,04,05,SE","PK,0K,01,02,03,04,05",...,COMMUNITY SCHOOL DISTRICT 15,"SKOP, ANITA",NYCDOE Borough Office - Brooklyn North,NULL,School Support Team 5- Brooklyn North,NULL,NULL,NULL,307,72
1,2020,17K002,K002,Parkside Preparatory Academy,331700010002,DOE,General Academic,Junior High-Intermediate-Middle,"06,07,08,SE","06,07,08",...,COMMUNITY SCHOOL DISTRICT 17,"ELLIS, CLARENCE",NYCDOE Borough Office - Brooklyn South,Mauriciere de Govia,School Support Team 2- Brooklyn South,NULL,NULL,NULL,309,71
2,2020,13K003,K003,P.S. 003 The Bedford Village,331300010003,DOE,General Academic,Elementary,"PK,0K,01,02,03,04,05,SE","PK,0K,01,02,03,04,05",...,COMMUNITY SCHOOL DISTRICT 13,"SAMUELS, KAMAR",NYCDOE Borough Office - Brooklyn North,NULL,School Support Team 3- Brooklyn North,NULL,NULL,NULL,303,79


The good news: the DBN is in system_code: 15K001 is the DBN format.

The surprise: this table has no zip code column. It’s not in the list, and primary_address_line_1 is only the street. But it does have census_tract, and that’s actually better for the Census join. A census tract is a small neighborhood, about 4,000 people, so the income data becomes more precise than by zip code.

To join on tract, you need two things: the tract number and the county. In NYC, each borough is its own county, and the borough is the letter in the DBN:

DBN letter	Borough	County code
M	Manhattan	061
X	Bronx	005
K	Brooklyn	047
Q	Queens	081
R	Staten Island	085

**Step 1: Look at the tract values first**

In [ ]:
snake(loc)[["system_code", "census_tract", "latitude", "longitude"]].head(10)

,system_code,census_tract,latitude,longitude
0,15K001,8000,40.648959,-74.01142
1,17K002,80200,40.656423,-73.951575
2,13K003,22700,40.682311,-73.955219
3,75K004,107800,40.6585,-73.879276
4,16K005,37700,40.685241,-73.92197
5,17K006,79400,40.648973,-73.956279
6,19K007,118201,40.689213,-73.872983
7,13K008,100,40.700709,-73.992841
8,13K009,20500,40.678554,-73.966333
9,15K010,14900,40.660912,-73.985946


Note what the tract numbers look like, for example 179, 17900 or 179.01. The Census writes tracts as 6 digits (017900), so you may need to convert them. Paste me a few values if you’re unsure.

**Step 2: Clean locations**

In [ ]:
def clean_locations(df):
    df = snake(df).rename(columns={"system_code": "dbn", "location_name": "school_name"})
    df["dbn"] = df["dbn"].str.strip().str.upper()

    # borough + county from the DBN letter (3rd character)
    letter = df["dbn"].str[2]
    df["borough"] = letter.map({"M": "Manhattan", "X": "Bronx", "K": "Brooklyn",
                                "Q": "Queens", "R": "Staten Island"})
    df["county"] = letter.map({"M": "061", "X": "005", "K": "047", "Q": "081", "R": "085"})

    # census tract as 6-digit text (adjust after Step 1 if needed)
    df["census_tract"] = df["census_tract"].astype(str).str.strip().str.zfill(6)

    # map coordinates as numbers
    df["latitude"]  = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

    # keep only what you need (drops phone, fax, principal names)
    keep = ["dbn", "school_name", "borough", "county", "census_tract",
            "location_category_description", "latitude", "longitude"]
    return df[keep].drop_duplicates(subset="dbn")

loc_clean = clean_locations(loc)
quality_report(loc_clean, "locations AFTER")
loc_clean.head()

--- locations AFTER ---
rows: 2131 | duplicates: 0
nulls per column:
 latitude     1
longitude    1
dtype: int64


,dbn,school_name,borough,county,census_tract,location_category_description,latitude,longitude
0,15K001,P.S. 001 The Bergen,Brooklyn,047,008000,Elementary,40.648959,-74.011420
1,17K002,Parkside Preparatory Academy,Brooklyn,047,080200,Junior High-Intermediate-Middle,40.656423,-73.951575
2,13K003,P.S. 003 The Bedford Village,Brooklyn,047,022700,Elementary,40.682311,-73.955219
3,75K004,P.S. K004,Brooklyn,047,107800,Elementary,40.658500,-73.879276
4,16K005,P.S. 005 Dr. Ronald McNair,Brooklyn,047,037700,Elementary,40.685241,-73.921970


Dropping the principal names and phone numbers follows the same privacy habit as removing SSN in your healthcare project.

Step 3: Download income by tract instead of zip

This replaces your zip-code income file. In notebook 1, add a new cell:

 this notebook hasn’t loaded the requests library yet. Add the imports at the top of the cell.

I also changed one line, the "in" part. The Census API expects state and county together in one string, separated by a space, which is safer than a list.
You should see about 2,300 rows with columns NAME, B19013_001E, state, county and tract, where the tract looks like 017900.

Tip: a NameError: name '...' is not defined almost always means a missing import, or a cell that hasn’t been run yet in this session. After a Colab restart, re-run your import cells first.

In [ ]:
import requests, pandas as pd
from google.colab import userdata

r = requests.get("https://api.census.gov/data/2022/acs/acs5", params={
    "get": "NAME,B19013_001E",
    "for": "tract:*",
    "in": "state:36 county:005,047,061,081,085",   # New York State, 5 NYC counties
    "key": userdata.get("census"),
}, timeout=60)

print(r.status_code, r.text[:200])

if r.text.strip().startswith("["):
    rows = r.json()
    income_tract = pd.DataFrame(rows[1:], columns=rows[0])
    income_tract.to_csv("income_tract_raw.csv", index=False)
    print(income_tract.shape)
    display(income_tract.head())

200 [["NAME","B19013_001E","state","county","tract"],
["Census Tract 1; Bronx County; New York","-666666666","36","005","000100"],
["Census Tract 2; Bronx County; New York","115064","36","005","000200"],

(2327, 5)


,NAME,B19013_001E,state,county,tract
0,Census Tract 1; Bronx County; New York,-666666666,36,005,000100
1,Census Tract 2; Bronx County; New York,115064,36,005,000200
2,Census Tract 4; Bronx County; New York,100553,36,005,000400
3,Census Tract 16; Bronx County; New York,41362,36,005,001600
4,Census Tract 19.01; Bronx County; New York,49500,36,005,001901


Next, clean the tract income and check that schools find their tract. Do these in notebook 2 (02_clean). If you downloaded the tract file in notebook 1, upload income_tract_raw.csv to notebook 2 first.

**1. Clean the tract income**

In [ ]:
income_tract_raw = pd.read_csv("income_tract_raw.csv", dtype=str)

def clean_income_tract(df):
    df = snake(df).rename(columns={"b19013_001e": "median_income"})
    df["county"] = df["county"].str.zfill(3)
    df["census_tract"] = df["tract"].str.zfill(6)
    df["median_income"] = pd.to_numeric(df["median_income"], errors="coerce")
    df.loc[df["median_income"] < 0, "median_income"] = np.nan   # -666666666 = no data
    return df[["county", "census_tract", "median_income"]]

income_tract = clean_income_tract(income_tract_raw)
quality_report(income_tract, "income by tract")
income_tract.head()

--- income by tract ---
rows: 2327 | duplicates: 0
nulls per column:
 median_income    126
dtype: int64


,county,census_tract,median_income
0,005,000100,NaN
1,005,000200,115064.0
2,005,000400,100553.0
3,005,001600,41362.0
4,005,001901,49500.0


In [ ]:
print("Locations:", loc_clean["census_tract"].head(10).tolist())
print("Census:   ", income_tract["census_tract"].head(10).tolist())

Locations: ['008000', '080200', '022700', '107800', '037700', '079400', '118201', '000100', '020500', '014900']
Census:    ['000100', '000200', '000400', '001600', '001901', '001902', '001903', '001904', '002001', '002002']


In [ ]:
test = loc_clean.merge(income_tract, on=["county", "census_tract"], how="left")
matched = test["median_income"].notna().mean()
print(f"Schools matched to an income value: {matched:.0%}")
test[["dbn", "school_name", "borough", "census_tract", "median_income"]].head(10)

Schools matched to an income value: 89%


,dbn,school_name,borough,census_tract,median_income
0,15K001,P.S. 001 The Bergen,Brooklyn,008000,75125.0
1,17K002,Parkside Preparatory Academy,Brooklyn,080200,95781.0
2,13K003,P.S. 003 The Bedford Village,Brooklyn,022700,122400.0
3,75K004,P.S. K004,Brooklyn,107800,75881.0
4,16K005,P.S. 005 Dr. Ronald McNair,Brooklyn,037700,77302.0
5,17K006,Norma Adams Clemons Academy,Brooklyn,079400,77153.0
6,19K007,P.S. 007 Abraham Lincoln,Brooklyn,118201,87180.0
7,13K008,P.S. 008 Robert Fulton,Brooklyn,000100,165188.0
8,13K009,P.S. 009 Teunis G. Bergen,Brooklyn,020500,149643.0
9,15K010,"Magnet School of Math, Science and Design Tech...",Brooklyn,014900,NaN


In [ ]:
loc_clean.to_parquet("locations_clean.parquet", index=False)
income_tract.to_parquet("income_tract_clean.parquet", index=False)